# 📈 IPO Prediction & Financial Analytics System

**A complete end-to-end Data Science project covering:**
- 🔧 Data Pipeline (ETL)
- 🔍 Exploratory Data Analysis (EDA)
- 🤖 Machine Learning Models (Regression + Classification)
- 📊 Interactive Visualizations
- 🌐 Streamlit Dashboard

**Data Sources:** SEBI Prospectuses (curated) + Yahoo Finance Official API  
**Technologies:** Python · Pandas · NumPy · Scikit-learn · XGBoost · Plotly · Streamlit · yfinance

---
> Built as part of Data Analyst Internship @ Bluestock.in | Apr–May 2026

## 🔧 Step 1: Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone your GitHub repository
# ⚠️ Replace YOUR_USERNAME with your actual GitHub username
!git clone https://github.com/YOUR_USERNAME/ipo-prediction-system.git
%cd ipo-prediction-system

In [ ]:
# Install all required packages
!pip install -r requirements.txt -q
print('✅ All packages installed successfully!')

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Imports successful!')

## 📂 Step 2: Load & Inspect the Dataset

In [ ]:
df = pd.read_csv('data/ipo_data.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
print('=== Statistical Summary ===')
df.describe().round(2)

## 🔧 Step 3: Data Pipeline — Clean & Feature Engineer

In [ ]:
# Run the data pipeline (skip yfinance for speed in Colab)
import sys
sys.path.insert(0, '.')
from utils.data_pipeline import run_pipeline

df_enriched = run_pipeline(skip_yfinance=True)
print(f'\nEnriched dataset shape: {df_enriched.shape}')
print(f'New columns added: {[c for c in df_enriched.columns if c not in df.columns]}')

In [ ]:
# Show listing gain category distribution
print('Listing Gain Category Distribution:')
print(df_enriched['listing_gain_category'].value_counts())

print('\nMarket Sentiment Distribution:')
print(df_enriched['market_sentiment'].value_counts())

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
df = df_enriched.copy()

# 4.1 Listing Gain Distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('IPO Listing Gain Distribution', fontsize=14, fontweight='bold')

gains = df['listing_gain_pct'].dropna()
axes[0].hist(gains, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(gains.mean(), color='orange', lw=2, ls='--', label=f'Mean: {gains.mean():.1f}%')
axes[0].axvline(gains.median(), color='green', lw=2, ls='--', label=f'Median: {gains.median():.1f}%')
axes[0].set_xlabel('Listing Gain (%)')
axes[0].set_ylabel('Number of IPOs')
axes[0].set_title('Distribution of Listing Gains')
axes[0].legend()

bins   = [-np.inf, -5, 0, 10, 30, 60, np.inf]
labels = ['< -5%', '-5-0%', '0-10%', '10-30%', '30-60%', '> 60%']
df['_cat'] = pd.cut(df['listing_gain_pct'], bins=bins, labels=labels)
counts = df['_cat'].value_counts().reindex(labels)
colors = ['#d32f2f','#ef9a9a','#ff9800','#2196F3','#4CAF50','#1B5E20']
bars   = axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white', alpha=0.85)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontsize=9)
axes[1].set_title('IPOs by Listing Gain Category')
axes[1].set_xlabel('Category')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')
df.drop(columns=['_cat'], inplace=True)
plt.tight_layout()
plt.savefig('assets/eda_listing_gain.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nMean Gain: {gains.mean():.2f}% | Median: {gains.median():.2f}% | Std: {gains.std():.2f}%')

In [ ]:
# 4.2 Sector Performance
sector_stats = (
    df.groupby('sector')['listing_gain_pct']
    .agg(['mean','count']).reset_index()
    .rename(columns={'mean':'avg_gain','count':'n'})
    .query('n >= 2').sort_values('avg_gain', ascending=True)
)

fig = px.bar(sector_stats, x='avg_gain', y='sector', orientation='h',
             color='avg_gain', color_continuous_scale='RdYlGn',
             text=[f'{v:.1f}% (n={n})' for v,n in zip(sector_stats['avg_gain'], sector_stats['n'])],
             title='Average IPO Listing Gain by Sector',
             labels={'avg_gain':'Avg Gain (%)','sector':'Sector'})
fig.update_layout(height=600, showlegend=False)
fig.show()
print('\nTop 5 Performing Sectors:')
print(sector_stats.tail(5)[['sector','avg_gain','n']].to_string(index=False))

In [ ]:
# 4.3 Subscription vs Listing Gain Correlation
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Subscription Analysis vs Listing Gain', fontsize=14, fontweight='bold')

sub_cols = [
    ('subscription_times', 'Overall Subscription (×)', 'steelblue'),
    ('qib_subscription',   'QIB Subscription (×)',     'purple'),
    ('retail_subscription','Retail Subscription (×)',   'green'),
]

for ax, (col, label, color) in zip(axes, sub_cols):
    valid = df[[col, 'listing_gain_pct']].dropna()
    cap   = valid[col].quantile(0.99)
    valid = valid[valid[col] <= cap]
    ax.scatter(valid[col], valid['listing_gain_pct'], alpha=0.5, s=25, color=color)
    z = np.polyfit(valid[col], valid['listing_gain_pct'], 1)
    xline = np.linspace(valid[col].min(), valid[col].max(), 100)
    ax.plot(xline, np.poly1d(z)(xline), color='orange', lw=2, ls='--')
    corr = valid[col].corr(valid['listing_gain_pct'])
    ax.set_title(f'{label}\n(r = {corr:.2f})')
    ax.set_xlabel(label)
    ax.set_ylabel('Listing Gain (%)')
    ax.axhline(0, color='gray', lw=1)

plt.tight_layout()
plt.savefig('assets/eda_subscription.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 4.4 Year-wise Trends
yearly = (
    df.groupby('year')
    .agg(count=('company_name','count'),
         avg_gain=('listing_gain_pct','mean'),
         pct_positive=('listing_gain_pct', lambda x: (x > 0).mean() * 100))
    .reset_index()
)

fig = make_subplots(rows=2, cols=1, subplot_titles=['IPO Volume by Year', 'Avg Gain & % Positive Listings'],
                    shared_xaxes=True, vertical_spacing=0.1)

fig.add_trace(go.Bar(x=yearly['year'], y=yearly['count'],
                     name='IPO Count', marker_color='steelblue', opacity=0.8), row=1, col=1)
fig.add_trace(go.Scatter(x=yearly['year'], y=yearly['avg_gain'],
                         name='Avg Gain %', mode='lines+markers',
                         line=dict(color='orange', width=2.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=yearly['year'], y=yearly['pct_positive'],
                         name='% Positive', mode='lines+markers',
                         line=dict(color='green', width=2.5, dash='dash')), row=2, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=1)
fig.update_layout(height=550, title='Year-wise IPO Market Trends')
fig.show()

In [ ]:
# 4.5 Correlation Heatmap
corr_cols = ['listing_gain_pct','issue_price','issue_size_cr','subscription_times',
             'qib_subscription','nii_subscription','retail_subscription','pe_ratio',
             'profit_cr','revenue_cr','debt_equity','promoter_holding_pct','nifty_on_listing']
corr_cols = [c for c in corr_cols if c in df.columns]
corr      = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, annot_kws={'size':8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('assets/eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop correlations with Listing Gain:')
corr_target = corr['listing_gain_pct'].drop('listing_gain_pct').sort_values(key=abs, ascending=False)
print(corr_target.round(3).to_string())

## 🤖 Step 5: Machine Learning Models

In [ ]:
from sklearn.model_selection   import train_test_split, cross_val_score
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.impute             import SimpleImputer
from sklearn.pipeline           import Pipeline
from sklearn.compose            import ColumnTransformer
from sklearn.ensemble           import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model       import LinearRegression, LogisticRegression
from sklearn.metrics            import mean_absolute_error, r2_score, mean_squared_error, accuracy_score, classification_report
from xgboost                    import XGBRegressor, XGBClassifier
import joblib, os

os.makedirs('models',  exist_ok=True)
os.makedirs('assets',  exist_ok=True)

NUMERIC_FEATURES = ['issue_price','issue_size_cr','subscription_times',
                    'qib_subscription','nii_subscription','retail_subscription',
                    'pe_ratio','revenue_cr','profit_cr','debt_equity',
                    'promoter_holding_pct','nifty_on_listing','year','month']
CATEGORICAL_FEATURES = ['sector','issue_type','market_sentiment']
TARGET = 'listing_gain_pct'

GAIN_BINS   = [-np.inf, 0, 10, 30, np.inf]
GAIN_LABELS = ['Negative','Flat (0-10%)','Moderate (10-30%)','Strong (>30%)']

# Prepare features
available_num = [c for c in NUMERIC_FEATURES     if c in df.columns]
available_cat = [c for c in CATEGORICAL_FEATURES if c in df.columns]
all_features  = available_num + available_cat

df_ml = df[all_features + [TARGET]].dropna(subset=[TARGET]).drop_duplicates()
X     = df_ml[all_features]
y_reg = df_ml[TARGET]
y_clf = pd.cut(y_reg, bins=GAIN_BINS, labels=GAIN_LABELS).astype(str)

X_train, X_test, y_tr, y_te, yc_tr, yc_te = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Target mean: {y_reg.mean():.2f}% | std: {y_reg.std():.2f}%')

In [ ]:
# Build preprocessor
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                     ('scaler',  StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                     ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([
    ('num', num_pipe, available_num),
    ('cat', cat_pipe, available_cat)
])

# Define models
reg_models = {
    'Linear Regression':     LinearRegression(),
    'Random Forest':         RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting':     GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=5, random_state=42),
    'XGBoost':               XGBRegressor(n_estimators=200, learning_rate=0.08, max_depth=5, random_state=42, verbosity=0),
}

print('=== REGRESSION RESULTS ===')
print(f'{"Model":<25} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
print('-' * 52)

reg_results = {}
reg_pipes   = {}

for name, model in reg_models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_tr)
    y_pred = pipe.predict(X_test)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = mean_squared_error(y_te, y_pred) ** 0.5
    r2   = r2_score(y_te, y_pred)
    reg_results[name] = {'MAE':mae, 'RMSE':rmse, 'R2':r2, 'preds':y_pred}
    reg_pipes[name]   = pipe
    print(f'{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2:>8.3f}')

best_reg_name = max(reg_results, key=lambda k: reg_results[k]['R2'])
print(f'\n✅ Best Model: {best_reg_name} (R²={reg_results[best_reg_name]["R2"]:.3f})')
joblib.dump(reg_pipes[best_reg_name], 'models/best_regressor.pkl')
print('   Saved → models/best_regressor.pkl')

In [ ]:
# 5.2 Model Comparison Chart
results_df = pd.DataFrame(reg_results).T[['MAE','RMSE','R2']].reset_index()
results_df.columns = ['Model','MAE','RMSE','R2']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Comparison — Regression Metrics', fontsize=14, fontweight='bold')
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0']

for ax, metric, better in zip(axes, ['MAE','RMSE','R2'], ['lower','lower','higher']):
    bars = ax.bar(results_df['Model'], results_df[metric], color=colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
               f'{val:.3f}', ha='center', fontsize=9)
    ax.set_title(f'{metric} ({better} is better)')
    ax.set_ylabel(metric)
    plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('assets/ml_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 5.3 Actual vs Predicted — Best Model
best_preds = reg_results[best_reg_name]['preds']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'{best_reg_name} — Prediction Quality', fontsize=13, fontweight='bold')

axes[0].scatter(y_te, best_preds, alpha=0.55, s=28, color='steelblue')
lims = [min(y_te.min(), best_preds.min()), max(y_te.max(), best_preds.max())]
axes[0].plot(lims, lims, color='orange', lw=1.8, ls='--', label='Perfect prediction')
axes[0].set_xlabel('Actual Listing Gain (%)')
axes[0].set_ylabel('Predicted Listing Gain (%)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

residuals = y_te.values - best_preds
axes[1].hist(residuals, bins=28, color='purple', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='orange', lw=2, ls='--')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('assets/ml_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 5.4 Feature Importance
best_model = reg_pipes[best_reg_name].named_steps['model']
if hasattr(best_model, 'feature_importances_'):
    cat_enc   = reg_pipes[best_reg_name].named_steps['preprocessor'].named_transformers_['cat']['encoder']
    cat_names = list(cat_enc.get_feature_names_out(available_cat))
    all_names = available_num + cat_names
    importances = best_model.feature_importances_
    idx     = np.argsort(importances)[::-1][:15]
    top_f   = [all_names[i] if i < len(all_names) else f'feat_{i}' for i in idx]
    top_imp = importances[idx]

    plt.figure(figsize=(10, 6))
    bars = plt.barh(top_f[::-1], top_imp[::-1], color='steelblue', alpha=0.85, edgecolor='white')
    plt.xlabel('Importance Score')
    plt.title(f'{best_reg_name} — Top 15 Feature Importances', fontsize=13, fontweight='bold')
    plt.grid(axis='x', alpha=0.4)
    plt.tight_layout()
    plt.savefig('assets/ml_feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('\nTop 5 Features:')
    for f, v in zip(top_f[:5], top_imp[:5]):
        print(f'  {f}: {v:.4f}')

In [ ]:
# 5.5 Classification Model
print('=== CLASSIFICATION RESULTS ===')
clf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, max_depth=8,
                                     random_state=42, class_weight='balanced'))
])
clf_pipe.fit(X_train, yc_tr)
yc_pred = clf_pipe.predict(X_test)
acc = accuracy_score(yc_te, yc_pred)
print(f'Accuracy: {acc*100:.1f}%')
print(classification_report(yc_te, yc_pred, zero_division=0))

joblib.dump(clf_pipe, 'models/best_classifier.pkl')
print('Saved → models/best_classifier.pkl')

## 🔮 Step 6: Predict a New IPO

In [ ]:
# Load best model and predict for a new IPO
model = joblib.load('models/best_regressor.pkl')

new_ipo = {
    'issue_price': 450,
    'issue_size_cr': 1800,
    'subscription_times': 75,
    'qib_subscription': 110,
    'nii_subscription': 85,
    'retail_subscription': 13,
    'pe_ratio': 32,
    'revenue_cr': 1200,
    'profit_cr': 95,
    'debt_equity': 0.25,
    'promoter_holding_pct': 68,
    'nifty_on_listing': 22000,
    'year': 2024,
    'month': 10,
    'sector': 'IT',
    'issue_type': 'OFS+Fresh',
    'market_sentiment': 'Bullish'
}

input_df = pd.DataFrame([new_ipo])
predicted_gain = float(model.predict(input_df)[0])
category = pd.cut([predicted_gain], bins=GAIN_BINS, labels=GAIN_LABELS)[0]

print('=' * 50)
print('IPO PREDICTION RESULT')
print('=' * 50)
print(f'Predicted Listing Gain : {predicted_gain:+.2f}%')
print(f'Category               : {category}')
print(f'Recommendation         : {"✅ Strong Buy" if predicted_gain > 30 else ("✅ Consider" if predicted_gain > 10 else ("⚠️ Neutral" if predicted_gain >= 0 else "❌ Avoid"))}')
print('=' * 50)

## 🌐 Step 7: Launch Streamlit Dashboard in Colab

In [ ]:
# Install localtunnel to expose Streamlit to internet
!npm install -q localtunnel
print('✅ localtunnel installed')

In [ ]:
# Get your Colab public IP (needed for localtunnel password)
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f'Your Colab IP (use as localtunnel password): {ip}')

In [ ]:
# Start Streamlit in background
import subprocess, time
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501',
     '--server.headless', 'true', '--server.enableCORS', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)
print('✅ Streamlit server started on port 8501')

In [ ]:
# Expose via localtunnel — public URL will be printed below
!npx localtunnel --port 8501
# 👆 Click the URL above to open the dashboard
# When prompted for password, enter the IP shown in the previous cell

---
## ✅ Project Summary

| Component | Status | Details |
|---|---|---|
| Data Pipeline | ✅ Done | 140+ IPOs cleaned, enriched, feature-engineered |
| EDA | ✅ Done | 8 charts — distribution, sector, subscription, heatmap |
| ML Regression | ✅ Done | 4 models — RF, GB, XGBoost, Linear |
| ML Classification | ✅ Done | RF + Logistic — 4 gain categories |
| Feature Importance | ✅ Done | Top drivers identified |
| Streamlit Dashboard | ✅ Done | 5-tab interactive dashboard |
| GitHub Ready | ✅ Done | Full repo with README, requirements.txt |

**Data Sources:** SEBI Prospectuses (official public documents) + Yahoo Finance API  
**Tech Stack:** Python · Pandas · NumPy · Scikit-learn · XGBoost · Plotly · Streamlit  

> 📌 For portfolio/resume — add your Streamlit Cloud link and GitHub link
